In [1]:
import pandas as pd
import numpy as np 


In [2]:
df  = pd.read_csv("DateFruit.csv", sep="\t")

In [3]:
df.head() #34 features and 1 caotegories

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-5.919126e+10,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-3.423307e+10,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-9.394835e+10,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-3.207431e+10,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-3.998097e+10,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df.shape


(898, 35)

In [5]:
# level Encoding ---converting the string values into numeric values 

X = df.drop("Class", axis=1)
y = df["Class"]

In [6]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [7]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
 
le = LabelEncoder()
y = le.fit_transform(y)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

y_train_encoded = le.fit_transform(y_train) 
y_test_encoded = le.transform(y_test)


In [9]:
scaler  = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Deep learning ANN

In [10]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader , TensorDataset

In [11]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_encoded, dtype = torch.long)

In [12]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [13]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset,batch_size = 32)

In [14]:
# Build our Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model  = nn.Sequential(
           nn.Linear(X.shape[1], 64), # 1st hidden layer
           nn.ReLU(), # 1st hidden layer
           nn.Linear(64,64),# 2nd hidden layer
           nn.ReLU(),# 2nd hidden layer
           nn.Linear(64,7), # final layer          
        )

    def forward(self, x):
        return self.model(x) 

In [15]:
model = ANN()

# loss & optim 
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [16]:
# Training the NN

epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criteria(outputs,yb)
        loss.backward()
        optimizer.step() # params update

        running_loss += loss.item()
        
    train_loss = running_loss / len(train_loader) # --loss in one epoch
    
    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/100, loss = 1.767110508421193
epoch = 2/100, loss = 1.1380096129749133
epoch = 3/100, loss = 0.7095731263575347
epoch = 4/100, loss = 0.5382104477156764
epoch = 5/100, loss = 0.4446678848370262
epoch = 6/100, loss = 0.3983240088690882
epoch = 7/100, loss = 0.35710986606452777
epoch = 8/100, loss = 0.32886490938456164
epoch = 9/100, loss = 0.2982009824203408
epoch = 10/100, loss = 0.2831607784913934
epoch = 11/100, loss = 0.2596170153954755
epoch = 12/100, loss = 0.23969115349261658
epoch = 13/100, loss = 0.22636590055797412
epoch = 14/100, loss = 0.21976893816305243
epoch = 15/100, loss = 0.2060749780224717
epoch = 16/100, loss = 0.1922562247061211
epoch = 17/100, loss = 0.19960318052250406
epoch = 18/100, loss = 0.19318278231050656
epoch = 19/100, loss = 0.17886926428131436
epoch = 20/100, loss = 0.16803885445646619
epoch = 21/100, loss = 0.16549382358789444
epoch = 22/100, loss = 0.15985047590473425
epoch = 23/100, loss = 0.15892207833087962
epoch = 24/100, loss = 0.1618672

In [24]:
# Evaluation 

model.eval()# this means our parameters are now fixed and from now no learning will be conducted

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)  # here the tensor values are produced like [0.1. 0.2 ,0.5, 1.3 .-0.5, ...] --> 7 values.
         # on these upper values we apply the softmax AF. 
        _, predicted = torch.max(outputs, 1) # this returns the max values and its index from the tensor arr.

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # actual samples in each batch
        
print(f"accuracy:  {correct/total*100}%")

accuracy:  94.44444444444444%
